# 03 추천 사이트 실행하기 (Colab)

순서대로 ▶ 버튼을 누르면 돼요.
1. 저장소 내려받기 (안 되면 `맛집앱.zip` 올리기)
2. 필요한 프로그램 설치
3. 사이트 켜기 → 나온 링크를 누르면 사이트가 열려요
4. (3번이 안 될 때만) 예비 방법
5. 끝낼 때만: 쌓인 리뷰 내려받기 (Colab은 끄면 파일이 사라져요)

⚠️ **'모두 실행'(Ctrl+F9) 말고 칸마다 ▶를 하나씩 눌러 주세요.**

## 1. 저장소 내려받기
깃허브 저장소를 Colab으로 내려받아요. 저장소가 비공개라서 실패하면 "파일 선택" 버튼이 나오니, 받아 둔 `맛집앱.zip`을 고르세요.

In [ ]:
import os, subprocess, zipfile
os.chdir('/content')
if not os.path.exists('/content/company-matjip'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/hevmadle/company-matjip'])
if os.path.exists('/content/company-matjip/app/app.py'):
    APP_DIR = '/content/company-matjip/app'
else:
    from google.colab import files
    up = files.upload()
    zipfile.ZipFile(list(up)[0]).extractall('/content')
    APP_DIR = '/content/맛집앱'
os.chdir(APP_DIR)
print('준비 완료:', APP_DIR)

## 2. 프로그램 설치
streamlit(사이트를 만드는 도구)을 설치해요.

In [ ]:
!pip install -q streamlit

## 3. 사이트 켜기
사이트를 켜고, 준비가 끝나면 링크를 보여 줘요. **이 링크는 Colab을 연 같은 크롬 창에서만 열려요.**
링크를 눌렀는데 빈 화면이면 5초쯤 기다렸다가 새로고침하세요.

In [ ]:
import os, subprocess, time, urllib.request
os.chdir(APP_DIR)

# 이미 켜져 있던 사이트가 있으면 끄고 새로 켠다
subprocess.run(['pkill', '-f', 'streamlit run'])
log = open('/content/streamlit_log.txt', 'w')
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0',
                  '--server.headless', 'true', '--server.enableCORS', 'false',
                  '--server.enableXsrfProtection', 'false'], stdout=log, stderr=log)

# 사이트가 실제로 준비될 때까지 기다린다(최대 60초)
ready = False
for _ in range(60):
    try:
        ready = urllib.request.urlopen('http://localhost:8501/_stcore/health', timeout=2).read() == b'ok'
    except Exception:
        pass
    if ready:
        break
    time.sleep(1)

if not ready:
    print('❌ 사이트가 켜지지 않았어요. 아래 내용을 복사해서 보내 주세요.')
    print(open('/content/streamlit_log.txt').read()[-2000:])
else:
    print('✅ 사이트가 켜졌어요.')
    from google.colab import output
    from google.colab.output import eval_js
    print('👉 이 주소를 누르세요(새 탭):', eval_js('google.colab.kernel.proxyPort(8501)'))
    print('   위 주소가 안 열리면 아래 링크를 눌러 보세요.')
    output.serve_kernel_port_as_window(8501, anchor_text='👉 (다른 방법) 맛집 추천 사이트 열기')

## 4. (예비) 3번 링크가 안 열릴 때만
다른 사람에게 보여 주거나, 3번이 안 될 때 쓰는 방법이에요. 인터넷 주소(`https://...trycloudflare.com`)를 만들어 줘요.
- 주소가 나온 뒤 **30초쯤 기다렸다가** 누르세요. 너무 빨리 누르면 '사이트에 연결할 수 없음'이 나오고, 그러면 주소창에서 새로고침하세요.
- 이 칸을 다시 실행하면 주소가 바뀌어요. 항상 마지막에 나온 주소를 쓰세요.

In [ ]:
import os, subprocess, time, re
os.chdir('/content')
if not os.path.exists('cloudflared'):
    subprocess.run('wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared', shell=True)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'])
tlog = open('/content/tunnel_log.txt', 'w')
subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8501'], stdout=tlog, stderr=tlog)

# 주소가 생기고, 연결이 실제로 등록될 때까지 기다린다(최대 60초)
url, registered = None, False
for _ in range(60):
    text = open('/content/tunnel_log.txt').read()
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
    url = found.group(0) if found else url
    registered = 'Registered tunnel connection' in text
    if url and registered:
        break
    time.sleep(1)

if url and registered:
    print('✅ 주소:', url)
    print('30초쯤 기다렸다가 누르세요. 안 열리면 새로고침하세요.')
else:
    print('❌ 주소를 만들지 못했어요. 아래 내용을 복사해서 보내 주세요.')
    print(open('/content/tunnel_log.txt').read()[-2000:])

## 5. 쌓인 리뷰·노출 기록 내려받기 (끝낼 때만)
⚠️ 예전에는 '모두 실행'을 누르면 이 칸 때문에 csv 파일이 내려받아졌어요. 그래서 평소에는 아무것도 안 하게 바꿨어요.
사이트를 다 쓰고 끝낼 때만 아래 `내려받기 = False`를 `True`로 바꾸고 실행하세요. 다음에 쓸 때는 이 파일을 `맛집앱/data/`에 넣으면 돼요.

In [ ]:
내려받기 = False   # 끝낼 때만 True로 바꾸고 실행

if 내려받기:
    from google.colab import files
    import os
    for f in [APP_DIR + '/data/employee_reviews.csv', APP_DIR + '/data/impressions.csv']:
        if os.path.exists(f):
            files.download(f)
else:
    print('내려받기는 건너뛰었어요. 끝낼 때만 True로 바꾸고 실행하세요.')